In [0]:
%pip install pystac-client shapely rasterio
dbutils.library.restartPython()

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

def find_repo_root(start_dir: str, marker_relative_path: str = "conf/ingestion_config.yaml", max_levels: int = 8) -> str:
    current = start_dir
    for _ in range(max_levels):
        candidate = os.path.join(current, marker_relative_path)
        if os.path.exists(candidate):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(
        f"Could not find '{marker_relative_path}' coming from {start_dir}"
    )

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

notebook_fs_path = f"/Workspace{notebook_path}" if not notebook_path.startswith("/Workspace") else notebook_path
start_dir = os.path.dirname(notebook_fs_path)

repo_root = find_repo_root(start_dir)
config_path = f"{repo_root}/conf/ingestion_config.yaml"

In [0]:
from src.stac_discovery import discover_scenes, index_scenes_to_delta
import yaml

with open(config_path) as f:
    config = yaml.safe_load(f)

active_aoi = config["stac_discovery"]["active_aoi"]
aoi_cfg = config["stac_discovery"]["aois"][active_aoi]


In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

raw_items = discover_scenes(
    catalog_url=config["cdse"]["stac_catalog_url"],
    collection=config["stac_discovery"]["collection"],
    bbox=aoi_cfg["bbox"],
    start_date=aoi_cfg["start_date"],
    end_date=aoi_cfg["end_date"],
)

index_scenes_to_delta(
    spark=spark,
    raw_items_dict=raw_items,
    target_table=config["unity_catalog"]["stac_source_table"],
    catalog_url=config["cdse"]["stac_catalog_url"],
    aoi_id=active_aoi,
)